<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/data_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

data_loader

In [1]:
import pandas as pd
import requests
import os
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive
from google.colab import files

# Mount Google Drive for data sharing
drive.mount('/content/drive')

def load_data():
    global dataset

    # STEP 1: Select data format
    data_format_label = widgets.Label("Select data format:")
    data_format_dropdown = widgets.Dropdown(
        options=[('CSV', 1), ('JSON', 2), ('API', 3)],
        description='Format:',
        disabled=False,
    )

    # STEP 2: Select data source
    data_source_label = widgets.Label("Select data source:")
    data_source_dropdown = widgets.Dropdown(
        options=[('File path', 1), ('Upload from computer', 2)],
        description='Source:',
        disabled=False,
    )

    file_path_text = widgets.Text(
        placeholder='Enter file path...',
        description='Path:',
        disabled=False
    )

    api_url_text = widgets.Text(
        placeholder='Enter API URL...',
        description='URL:',
        disabled=False
    )

    upload_button = widgets.FileUpload(
        description='Upload file',
        multiple=False
    )

    load_button = widgets.Button(description="Load Data")
    output_area = widgets.Output()

    # Hide all fields initially
    file_path_text.layout.display = 'none'
    api_url_text.layout.display = 'none'
    upload_button.layout.display = 'none'
    data_source_dropdown.layout.display = 'none'

    def on_format_change(change):
        data_source_dropdown.layout.display = 'none'
        file_path_text.layout.display = 'none'
        upload_button.layout.display = 'none'
        api_url_text.layout.display = 'none'

        if change['new'] == 1 or change['new'] == 2:  # CSV or JSON
            data_source_dropdown.layout.display = 'block'
        elif change['new'] == 3:  # API
            api_url_text.layout.display = 'block'

    def on_source_change(change):
        file_path_text.layout.display = 'none'
        upload_button.layout.display = 'none'

        if change['new'] == 1:  # File path
            file_path_text.layout.display = 'block'
        elif change['new'] == 2:  # Upload
            upload_button.layout.display = 'block'

    data_format_dropdown.observe(on_format_change, names='value')
    data_source_dropdown.observe(on_source_change, names='value')

    def on_load_button_click(b):
        with output_area:
            output_area.clear_output()
            selected_format = data_format_dropdown.value

            try:
                # Create ml_project directory if it doesn't exist
                os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)

                if selected_format == 1:  # CSV
                    if data_source_dropdown.value == 1:  # File path
                        file_path = file_path_text.value
                        if not os.path.exists(file_path):
                            raise FileNotFoundError(f"File does not exist: {file_path}")
                        dataset = pd.read_csv(file_path)
                    else:  # Upload
                        if not upload_button.value:
                            raise ValueError("Please select a file to upload")
                        uploaded_file = list(upload_button.value.values())[0]
                        content = uploaded_file['content']
                        import io
                        dataset = pd.read_csv(io.BytesIO(content))

                    dataset.to_csv('/content/drive/MyDrive/ml_project/dataset.csv', index=False)
                    print("CSV data loaded and saved to Google Drive!")
                    print(dataset.head())

                elif selected_format == 2:  # JSON
                    if data_source_dropdown.value == 1:  # File path
                        file_path = file_path_text.value
                        if not os.path.exists(file_path):
                            raise FileNotFoundError(f"File does not exist: {file_path}")
                        dataset = pd.read_json(file_path)
                    else:  # Upload
                        if not upload_button.value:
                            raise ValueError("Please select a file to upload")
                        uploaded_file = list(upload_button.value.values())[0]
                        content = uploaded_file['content']
                        import io
                        dataset = pd.read_json(io.BytesIO(content))

                    dataset.to_csv('/content/drive/MyDrive/ml_project/dataset.csv', index=False)
                    print("JSON data loaded and saved to Google Drive!")
                    print(dataset.head())

                elif selected_format == 3:  # API
                    api_url = api_url_text.value
                    response = requests.get(api_url, timeout=20)
                    response.raise_for_status()
                    api_data = response.json()

                    print(f"Data type: {type(api_data)}")

                    if isinstance(api_data, dict):
                        dataset = pd.DataFrame([api_data])
                    elif isinstance(api_data, list):
                        dataset = pd.DataFrame(api_data)
                    else:
                        raise ValueError("Unsupported data format from API")

                    dataset.to_csv('/content/drive/MyDrive/ml_project/dataset.csv', index=False)
                    print("API data loaded and saved to Google Drive!")
                    print(dataset.head())

            except Exception as e:
                print(f"Error: {e}")

    load_button.on_click(on_load_button_click)

    # Display all widgets
    display(
        widgets.VBox([
            data_format_label,
            data_format_dropdown,
            data_source_label,
            data_source_dropdown,
            file_path_text,
            api_url_text,
            upload_button,
            load_button,
            output_area
        ])
    )

Mounted at /content/drive


In [2]:
print("\n1. Loading data...")
load_data()


1. Loading data...
